**Few Points**
1.During training phase if the chance is waste means no changes between to states will result in reward=-100

2. When the highest tile is in any corner of board reward =+2 in training phase



3.Used Epsilon decay for traininh and epsilon greedy for eval part .

4. this improved the score and consicutively reached 1100-1200+

5.Earlier was using the epsilon greedy for the train phase but to balance between the exploration and exploitation part then applied the decay


In [1]:
pip install gymnasium-2048


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


In [2]:

import gymnasium as gym
import gymnasium_2048
import numpy as np
import random

def parse_board(board):
    b = np.array(board, dtype=float).squeeze()

    if b.ndim == 3:
        axis = 0 if b.shape[0] > b.shape[-1] else -1
        b = np.argmax(b, axis=axis).astype(float)
        b = np.where(b > 0, 2 ** b, 0)
    elif b.ndim == 1 and b.size == 16:
        b = b.reshape((4, 4))

    return b

class FeatureExtractor2048:
    def __init__(self, num_actions=4):
        self.num_actions = num_actions
        self.num_state_features = 5

    def total_features(self):
        return self.num_state_features * self.num_actions

    def _get_state_features(self, board):
        board = parse_board(board)

        board_log = np.zeros_like(board)
        mask = board > 0
        board_log[mask] = np.log2(board[mask])

        empty_cells = np.sum(board == 0) / 16.0

        max_tile = np.max(board_log) / 11.0

        max_pos = np.unravel_index(np.argmax(board), board.shape)
        corner_max = 1.0 if max_pos in [(0,0),(0,3),(3,0),(3,3)] else 0.0

        sum_tiles = np.sum(board_log) / 50.0

        mono = 0
        for row in range(4):
            for col in range(3):
                if board_log[row, col] >= board_log[row, col+1]:
                    mono += 1
        for col in range(4):
            for row in range(3):
                if board_log[row, col] >= board_log[row+1, col]:
                    mono += 1
        monotonicity = mono / 24.0

        return [empty_cells, max_tile, corner_max, sum_tiles,monotonicity]

    def extract(self, board, action):
        state_features = self._get_state_features(board)
        feature_vector = []
        for a in range(self.num_actions):
            if a == action:
                feature_vector.extend(state_features)
            else:
                feature_vector.extend([0.0] * self.num_state_features)
        return np.array(feature_vector)


class LinearQAgent2048:
    def __init__(self, alpha=0.005, gamma=0.95, epsilon=0.5, epsilon_min=0.01, epsilon_decay=0.995):
        self.features      = FeatureExtractor2048()
        self.weights       = np.zeros(self.features.total_features())
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.num_actions   = 4

    def get_q_value(self, board, action):
        """Q(s, a) = features(s, a) · weights"""
        return np.dot(self.features.extract(board, action), self.weights)

    def choose_action(self, board, is_training=True):
        if is_training and random.random() < self.epsilon:
            return random.randint(0, 3)
        return int(np.argmax([self.get_q_value(board, a) for a in range(self.num_actions)]))

    def update(self, board, action, reward, next_board, done):
        """SGD weight update using the TD error."""
        current_q  = self.get_q_value(board, action)
        max_next_q = 0.0 if done else max(self.get_q_value(next_board, a) for a in range(self.num_actions))
        td_error = reward + self.gamma * max_next_q - current_q
        self.weights += self.alpha * td_error * self.features.extract(board, action)

def train(episodes=500):
    env   = gym.make('gymnasium_2048/TwentyFortyEight-v0')
    agent = LinearQAgent2048()
    for episode in range(episodes):
        board, _ = env.reset()
        if isinstance(board, tuple):
            board = board[0]
        done         = False
        total_reward = 0
        peak_tile    = 0
        while not done:
            action = agent.choose_action(board, is_training=True)
            next_board, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            if np.array_equal(parse_board(board), parse_board(next_board)):
                reward = -100.0
            b = parse_board(next_board)
            max_pos = np.unravel_index(np.argmax(b), b.shape)
            if max_pos in [(0,0),(0,3),(3,0),(3,3)]:
                reward += 2.0
            agent.update(board, action, reward, next_board, done)
            board        = next_board
            total_reward += reward
            peak_tile    = max(peak_tile, int(np.max(parse_board(board))))
        agent.epsilon = max(agent.epsilon_min, agent.epsilon * agent.epsilon_decay)
        # if (episode + 1) % 50 == 0:
        #     print(f"Ep {episode+1}/{episodes} | Reward: {total_reward:.1f} | Peak Tile: {peak_tile} | Eps: {agent.epsilon:.3f}")
    env.close()
    return agent

def evaluate(agent, games=5, max_steps=1000):
    env = gym.make('gymnasium_2048/TwentyFortyEight-v0', render_mode="human")
    for game in range(games):
        board, _ = env.reset()
        done  = False
        steps = 0
        score = 0.0
        while not done and steps < max_steps:
            prev_board = parse_board(board)
            action = agent.choose_action(board, is_training=False)
            board_raw, reward, terminated, truncated, _ = env.step(action)
            if np.array_equal(prev_board, parse_board(board_raw)):
                action = random.randint(0, 3)
                board_raw, reward, terminated, truncated, _ = env.step(action)
            board = board_raw
            done   = terminated or truncated
            score += reward
            steps += 1
        max_tile = int(np.max(parse_board(board)))
        print(f"Game {game+1}  Score: {score:.0f}  Max Tile: {max_tile}  Steps: {steps}")
    env.close()

if __name__ == "__main__":
    trained_agent = train(episodes=500)
    evaluate(trained_agent, games=5)


Game 1  Score: 1012  Max Tile: 64  Steps: 137
Game 2  Score: 444  Max Tile: 32  Steps: 82
Game 3  Score: 688  Max Tile: 64  Steps: 100
Game 4  Score: 568  Max Tile: 64  Steps: 84
Game 5  Score: 1768  Max Tile: 128  Steps: 180
